## Programming Assignment #3

**Natural Language Processing**

100 points possible.

This assignment asks you to perform specific data cleaning tasks.

# Part 0 -- Submission Details


(10 points) Please enter your name and the date below. Submit your answers as a completed notebook by the deadline posted on Canvas.  Late submissions will not get credit for this section.

Name: Valeria Calderon Triana

Date:11/16/2025


# Phase 1 -- Setup and Data Collection

(30 points) InstructionsSetup and Libraries: 

Import all necessary libraries: pandas, numpy, re, string, nltk (for processing), and the required modules from sklearn (for splitting, feature extraction, and modeling). Also, use nltk.download() to ensure you have the required resources like 'punkt', 'stopwords', and 'wordnet' downloaded.

Data Loading and Label Encoding: Load the movie review dataset (assuming it is saved as 'IMDB_Dataset.csv'). The sentiment labels usually come as strings ('positive', 'negative'); map these to a binary numeric format (1 for positive, 0 for negative).

Data Splitting: Separate the text reviews ($X$) from the binary sentiment labels ($y$). Use sklearn.model_selection.train_test_split to create your working sets: $X_{train}$, $X_{test}$, $y_{train}$, and $y_{test}$. Use a test size of 0.2 (20%) and ensure the split is stratified to maintain the same proportion of positive/negative reviews in both sets.

Based on the above instructions, make the following cell complete.

In [1]:
# CODE CELL FOR PHASE 1: SETUP AND DATA ACQUISITION (Streamlined)

import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Download necessary NLTK data (Using the simple call now that path is set)
try:
    nltk.download("punkt")
    nltk.download("stopwords")
    nltk.download("wordnet")
    print("Resources downloaded/verified successfully.")
except Exception as e:
    print(f"Error during download: {e}")

# Load the dataset
df = pd.read_csv("IMDB_Dataset.csv")

# Convert sentiment labels to binary (1 for positive, 0 for negative)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("Data Loaded. Shape:", df.shape)

# 1.3 Data Splitting (80% Train, 20% Test)
X = df['review']   # Text features
y = df['sentiment']  # Target labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]} reviews")
print(f"Testing set size: {X_test.shape[0]} reviews")



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vtriana\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vtriana\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vtriana\AppData\Roaming\nltk_data...


Resources downloaded/verified successfully.
Data Loaded. Shape: (499, 2)

Training set size: 399 reviews
Testing set size: 100 reviews


# Phase 2 -- Data Preprocessing and Feature Engineering

(40 points) This phase is crucial for transforming the raw text into the numerical $\text{TF-IDF}$ vectors required by the machine learning algorithm.

Instructions

Define Preprocessing Function: Create a function, preprocess_text(text), that applies the following transformations in order:
        Lowercase Conversion: Convert the entire text to lowercase.
        Punctuation and Number Removal: Use Python's string methods (str.maketrans) and regular expressions (re) to strip punctuation and numbers.
        Tokenization: Split the cleaned text into a list of individual tokens.
        Stop Word Filtering: Remove common English stop words using the nltk.corpus.stopwords list.
        Lemmatization: Apply WordNetLemmatizer to reduce tokens to their base form.
        Output: Rejoin the processed tokens into a single string to serve as the input for the vectorizer.

Apply Preprocessing: Apply the preprocess_text function to both the training ($X_{train}$) and testing ($X_{test}$) text sets.

TF-IDF Vectorization:
    Initialize the TfidfVectorizer (set max_features=5000 to limit the vocabulary size).
    The core step: Fit and transform the vectorizer only on the training data ($\mathbf{X}_{train\_cleaned}$). This step learns the vocabulary and calculates the $\text{IDF}_j$ weights for the corpus.
    Transform the testing data ($\mathbf{X}_{test\_cleaned}$) using the fitted vectorizer. This ensures the test set uses the same features and $\text{IDF}$ weights derived from the training corpus. The output of this step will be the final sparse $\mathbf{TF\text{-}IDF}$ matrices: $\mathbf{X}_{train\_tfidf}$ and $\mathbf{X}_{test\_tfidf}$.

    Based on the above instructions, make the following cell complete.

In [2]:
# CODE CELL FOR PHASE 2: DATA PREPROCESSING AND FEATURE ENGINEERING
import nltk
nltk.download('punkt_tab')

# Initialize Lemmatizer and Stopwords list
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase conversion & Punctuation/Number removal
    text = text.lower()                                          # lowercase conversion
    text = re.sub(r'\d+', '', text)                              # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    
    # 2. Tokenization
    tokens = text.split()
    
    # 3. Stop Word Removal and Lemmatization
    cleaned_tokens = []
    for word in tokens:
        if word not in stop_words:
            # Lemmatize the word
            word = lemmatizer.lemmatize(word)
            cleaned_tokens.append(word)
            
    # Join tokens back into a single string for TfidfVectorizer input
    return ' '.join(cleaned_tokens)

# Apply the cleaning function to the datasets
X_train_cleaned = X_train.apply(preprocess_text)
X_test_cleaned = X_test.apply(preprocess_text)

print("Text Preprocessing Complete.")

# 3. TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit on training data to learn vocabulary and IDF weights, then transform
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_cleaned)

# Transform test data using the fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test_cleaned)

print("\nTF-IDF Vectorization Complete.")
print(f"Training TF-IDF Matrix Shape: {X_train_tfidf.shape}")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vtriana\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Text Preprocessing Complete.

TF-IDF Vectorization Complete.
Training TF-IDF Matrix Shape: (399, 5000)


# Phase 3 -- Model Training and Evaluation

(30 points) This final phase trains the classifier and validates its performance using standard metrics.

Instructions

Model Training (Supervised Learning): Instantiate a Logistic Regression classifier. Train the model by calling the .fit() method on your $\mathbf{X}_{train\_tfidf}$ features and $\mathbf{y}_{train}$ labels. This process learns the feature weights $\theta$.

Prediction: Use the trained model's .predict() method on the unseen test features ($\mathbf{X}_{test\_tfidf}$) to generate the predicted labels ($\mathbf{y}_{pred}$). The model predicts the class with the highest score $\Psi_{xy}$.

Model Evaluation: Calculate and display the following metrics using sklearn.metrics:  Accuracy Score. Classification Report (showing Precision, Recall, and F1-Score for both classes). Confusion Matrix.

Interpretation: Extract the learned coefficients from the trained model. Identify the words (features) that correspond to the top 10 largest positive coefficients (strongest indicators of 'positive' sentiment) and the top 10 largest negative coefficients (strongest indicators of 'negative' sentiment).

   What you need to do: below are complete code, please provide annotations to explain the major steps. You may investigate major functions used and explain them.

In [3]:
# CODE CELL FOR PHASE 3: MODEL TRAINING AND EVALUATION
"""
 3.1 Model Training
Logistic regression is a linear classifier that is suitable for modeling binary sentiment.
The solver libnear is used for smaller datasets, and the random state ensures that the results are reproducible.
 """
model = LogisticRegression(solver='liblinear', random_state=42)
"""
# Train the model
the model on the TF_IDF learnes a weight for each feature that has a positive or negative sentiment.
"""
model.fit(X_train_tfidf, y_train)

print("Logistic Regression Model Training Complete.")

# 3.2 Prediction
"""
the model.predict() applies the learned weights form the model to the new data to predict the sentiment
"""
y_pred = model.predict(X_test_tfidf)

# 3.3 Model Evaluation
print("\n--- Model Performance Evaluation ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}") # the accuracy score is the porcentage of correct predictions  
#classification report shows the precision, recall and F1 score
# so precision being =TP/(TP/FP)
#recall TP  /(TP+FN)
#F1 is the harmonic mean of precision and recall.
print("\nClassification Report (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
print("\nConfusion Matrix:")
#the confusion matrix shows the true vs predicted results.
print(confusion_matrix(y_test, y_pred))

# 3.4 Interpretation: Top Predictors (Weights)
'''
Each of the TF-IDF features that has a coeffcient in the regression model
the positive weight pushes prediction towards positive
and the negative towards negative.
'''
#it gets all the feature names in the same order as the coefficients
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
#this model has only one coefficient vector since it was binary
coefficients = model.coef_[0]

# in here we combine the features with their weights.
weights_df = pd.DataFrame({'feature': feature_names, 'weight': coefficients})

# Top 10 Positive Predictors
#this just sorts the weight and the ones that have a higher positive sentiment are printed
top_positive = weights_df.sort_values(by='weight', ascending=False).head(10)
print("\n--- Top 10 Words Predicting POSITIVE Sentiment ---")
print(top_positive)

# Top 10 Negative Predictors
#this just sorts the weight and the ones that have a higher negative sentiment are printed
top_negative = weights_df.sort_values(by='weight', ascending=True).head(10)
print("\n--- Top 10 Words Predicting NEGATIVE Sentiment ---")
print(top_negative)

Logistic Regression Model Training Complete.

--- Model Performance Evaluation ---
Accuracy Score: 0.7700

Classification Report (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

    Negative       0.73      0.91      0.81        53
    Positive       0.85      0.62      0.72        47

    accuracy                           0.77       100
   macro avg       0.79      0.76      0.76       100
weighted avg       0.79      0.77      0.76       100


Confusion Matrix:
[[48  5]
 [18 29]]

--- Top 10 Words Predicting POSITIVE Sentiment ---
        feature    weight
1896      great  1.255985
4225      still  0.700448
1483  excellent  0.686986
1776       game  0.641606
4923      world  0.606699
2673       love  0.593278
4242      story  0.586502
4972      young  0.571853
3093      often  0.537558
2636     little  0.520131

--- Top 10 Words Predicting NEGATIVE Sentiment ---
       feature    weight
334        bad -1.488192
4931     worst -0.951959
4443  terr